# 00 Data Overview And Split Check

This notebook checks the hourly NL day-ahead target series before any forecasting model is evaluated.

What this notebook covers:
- whether the cleaned pipeline truly delivers UTC timestamps
- where missing hourly target values still exist on the canonical UTC grid
- how the train, validation, and test periods are defined
- why split windows are stated in local delivery dates even though stored timestamps remain UTC

Leakage prevention:
- timestamps are stored in UTC inside the pipeline
- local time is only used to define business rules such as delivery days and the 08:00 D-1 forecast origin
- the walk-forward engine only uses history strictly before each forecast origin

In [ ]:
from pathlib import Path
import pandas as pd

run_root = Path('data/02_Forecasting/01_DA_prices/hourly_da/runs')
latest_run = sorted(run_root.glob('*_data_overview'))[-1]
latest_run

In [ ]:
timezone_audit = pd.read_json(latest_run / 'timezone_audit.json', typ='series')
gap_summary = pd.read_csv(latest_run / 'gap_summary.csv')
gap_intervals = pd.read_csv(latest_run / 'gap_intervals.csv')
split_summary = pd.read_csv(latest_run / 'split_summary.csv')

display(timezone_audit)
display(gap_summary)
display(split_summary)

Interpretation notes:

- `UTC_CONFIRMED` means the cleaned hourly file parses as timezone-aware UTC and carries explicit UTC offsets.
- A non-zero missing-hour count does **not** mean the timezone is wrong. It means the underlying published source series contains gaps.
- The forecasting pipeline keeps those gaps explicit on the canonical hourly UTC grid and only scores rows with observed `y_true` values.

In [ ]:
gap_intervals.head(20)